In [1]:
import pandas as pd 
from dotenv import load_dotenv
import os
from pathlib import Path
import xlrd
import numpy as np
import re

load_dotenv(override=True)

LEDGER_DIR = os.getenv("ledger_dir")

def process_bank_file(file):
    df = pd.read_excel(file, engine='xlrd')

    #--- Extract branch code and name ---

    muthoot_col = df.columns[df.columns.to_series().str.contains("muthoot", case=False, na=False)]
    dim_rec = {}
    if len(muthoot_col) > 0:
        df = df.rename(columns={muthoot_col[0]: "source_dim_col"})

        match = re.search(
            r"Branch:\s+(.+?)-(\d+)\s*$",
            muthoot_col[0], re.IGNORECASE | re.MULTILINE
        )

        dim_rec = {
            "branch_code": match.group(2) if match else None,
            "branch_name": re.sub(r"\s*-\s*", " ", match.group(1)).strip() if match else None,
            "acc_no":      None  # not present in this format
        }

    #--- Extract account number ---
    keyword = "Account:"

    matches = (
        df.apply(lambda col: col.astype(str)
                .str.extract(rf"^({keyword}.+)", expand=False))
        .stack()
        .dropna()
        .unique()
    )

    if len(matches) > 0:
        match = re.search(r"-(\d+)$", matches[0]) if matches.size > 0 else None
        dim_rec["acc_no"] = match.group(1) if match else None

    #--- Extract opening balance ---
        
    opening_mask = (
        df.astype(str)
          .apply(lambda col: col.str.contains("opening", case=False, na=False))
          .any(axis=1)
    )
    opening_rows = (
            df[opening_mask]
            .replace(r"^\s*$", np.nan, regex=True)
            .dropna(axis=1, how="all")
        )

    if not opening_rows.empty and opening_rows.shape[1] > 1:
        raw_val = opening_rows.iloc[0, 4]   # column[4] holds the value
        dim_rec["opening_balance"] = pd.to_numeric(
            str(raw_val).replace(",", ""), errors="coerce"
        )
    else:
        dim_rec["opening_balance"] = None

    #--- Clean the dataframe ---
    df = df.replace(r"^\s*$", np.nan, regex=True)
    df = df.dropna(axis=1, how="all")

    pattern = "total|opening|balance as on|page|Ledger|account:"
    df = df[~df.apply(lambda x: x.astype(str).str.contains(pattern, case=False).any(), axis=1)]
    df = df.reset_index(drop=True)

    meta_keys = ["branch_code", "branch_name", "acc_no"]
    for key in meta_keys:
        df[key] = dim_rec.get(key)


    return df, dim_rec
    


In [2]:
ledger_df = []
dim_records = []
for file in Path(LEDGER_DIR).glob("*.xls"):
    df, dim_rec = process_bank_file(file)

    if df is not None:
        ledger_df.append(df)
    if dim_rec is not None:
        dim_records.append(dim_rec)

ledger_df = pd.concat(ledger_df, ignore_index=True)
dim_rec_df = pd.DataFrame(dim_records)
dim_rec_df

C:\Users\user\AppData\Local\Temp\ipykernel_15892\3706911472.py:58: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(r"^\s*$", np.nan, regex=True)
C:\Users\user\AppData\Local\Temp\ipykernel_15892\3706911472.py:58: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace(r"^\s*$", np.nan, regex=True)
C:\Users\user\AppData\Local\Temp\ipykernel_15892\3706911472.py:58: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_obj

,branch_code,branch_name,acc_no,opening_balance
0,0182,TIRUPUR LAKSHMI NAGAR,40168221385,353857.60
1,0627,HYDERABAD VANASTALIPURAM,43143793734,2759866.14
2,1046,UTHUKOTTAI,43782239368,2092603.80
3,4304,GAJWEL (AP),33069117110,400014.04


In [3]:
# Find header row, slice data, set columns and clean in one chain
mask = ledger_df.astype(str).apply(
    lambda col: col.str.contains("date", case=False, na=False)
)

header_row_idx = mask.any(axis=1).idxmax()

df = ledger_df.iloc[header_row_idx + 1:].copy()
df.columns = ledger_df.iloc[header_row_idx]
df = df.reset_index(drop=True)

 

In [4]:
df[["balance_amount", "balance_type"]] = (
    df["Balance"]
      .astype(str)
      .str.strip()
      .str.extract(r"([\d,]+\.?\d*)\s*(Cr|Dr)", flags=re.IGNORECASE)
)

df["balance_amount"] = pd.to_numeric(
    df["balance_amount"].str.replace(",", "", regex=False),
    errors="coerce"
)

df["balance_type"] = df["balance_type"].str.upper()   # normalize → CR / DR

df["cust_name"] = None

df = df.drop(columns=["Balance", "Particulars"], errors="ignore")   

df.head(10)

3,Date,Vh.No,NaN,NaT,Type,Chq. No.,NaN,NaN,Debit,Credit,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,balance_amount,balance_type,cust_name
0,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None
1,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None
2,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None
3,2026-03-02 00:00:00,NaN,1.0,NaT,BV,189263,NaN,Cheque :MAHALAKSHMI MAHALAKSHMI#BTB#BTB8-TN17#...,NaN,"3,82,075.00",NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,28217.4,CR,None
4,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None
5,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None
6,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None
7,2026-03-02 00:00:00,NaN,2.0,NaT,BV,NaN,NaN,EXCESS CASH SENT TO OUR BANK ACCOUNT THROUGH M...,"10,00,000.00",NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,971782.6,DR,None
8,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None
9,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,NaN,NaN,None


In [5]:
list(enumerate(df.columns))

[(0, 'Date'),
 (1, 'Vh.No'),
 (2, np.float64(nan)),
 (3, NaT),
 (4, 'Type'),
 (5, 'Chq. No.'),
 (6, nan),
 (7, nan),
 (8, 'Debit'),
 (9, 'Credit'),
 (10, nan),
 (11, '0182'),
 (12, 'TIRUPUR LAKSHMI NAGAR'),
 (13, '40168221385'),
 (14, 'balance_amount'),
 (15, 'balance_type'),
 (16, 'cust_name')]

In [6]:
#--- Standardize column names ---
df.columns = (
    df.columns
      .str.strip()                          
      .str.lower()                          
      .str.replace(r"[^\w\s]", "", regex=True)   
      .str.replace(r"\s+", "_", regex=True)      
)
col_mapping = {
    "date"    : "txn_date",
    # "vhno"    : "vh_no",
    "debit"   : "debit_amount",
    "credit"  : "credit_amount",
    # "balance"    : "balance_amount",       
}
df.columns = [
    next((new for key, new in col_mapping.items() if key in str(col).lower()), col)
    for col in df.columns
]

# Convert to datetime first, then drop invalid dates
df["txn_date"] = pd.to_datetime(df["txn_date"], errors="coerce")
df = df[df["txn_date"].notna()].reset_index(drop=True)

cols = df.columns.tolist()
cols[2] = "vh_no"
cols[7] = "particulars"
cols[11] = "branch_code"
cols[12] = "branch_name"
cols[13] = "acc_no"

df.columns = cols



list(enumerate(df.columns))

[(0, 'txn_date'),
 (1, 'vhno'),
 (2, 'vh_no'),
 (3, NaT),
 (4, 'type'),
 (5, 'chq_no'),
 (6, nan),
 (7, 'particulars'),
 (8, 'debit_amount'),
 (9, 'credit_amount'),
 (10, nan),
 (11, 'branch_code'),
 (12, 'branch_name'),
 (13, 'acc_no'),
 (14, 'balance_amount'),
 (15, 'balance_type'),
 (16, 'cust_name')]

In [7]:

# Convert numeric columns
cols = ['credit_amount', 'debit_amount', 'balance_amount']
for col in cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    ).fillna(0)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202 entries, 0 to 201
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   txn_date        202 non-null    datetime64[ns]
 1   vhno            0 non-null      object        
 2   vh_no           202 non-null    float64       
 3   NaT             0 non-null      datetime64[ns]
 4   type            202 non-null    object        
 5   chq_no          110 non-null    object        
 6   nan             0 non-null      object        
 7   particulars     202 non-null    object        
 8   debit_amount    202 non-null    float64       
 9   credit_amount   202 non-null    float64       
 10  nan             0 non-null      object        
 11  branch_code     202 non-null    object        
 12  branch_name     202 non-null    object        
 13  acc_no          202 non-null    object        
 14  balance_amount  202 non-null    float64       
 15  balanc

In [8]:
df["txn_type"] = np.select(
    [
        df["credit_amount"] > 0,
        df["debit_amount"] > 0
    ],
    ["cr", "dr"],
    default=None
)

df.head()

,txn_date,vhno,vh_no,NaT,type,chq_no,NaN,particulars,debit_amount,credit_amount,NaN,branch_code,branch_name,acc_no,balance_amount,balance_type,cust_name,txn_type
0,2026-03-02,NaN,1.0,NaT,BV,189263,NaN,Cheque :MAHALAKSHMI MAHALAKSHMI#BTB#BTB8-TN17#...,0.0,382075.0,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,28217.4,CR,None,cr
1,2026-03-02,NaN,2.0,NaT,BV,NaN,NaN,EXCESS CASH SENT TO OUR BANK ACCOUNT THROUGH M...,1000000.0,0.0,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,971782.6,DR,None,dr
2,2026-03-03,NaN,4.0,NaT,BV,189265,NaN,TAKEOVER CASH SENT TO VALAYANKADU BRANCH CUSTO...,0.0,35040.0,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,936742.6,DR,None,cr
3,2026-03-03,NaN,5.0,NaT,BV,189264,NaN,TAKEOVER CASH SENT TO VALAYANKADU BRANCH CUSTO...,0.0,62950.0,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,873792.6,DR,None,cr
4,2026-03-03,NaN,6.0,NaT,BV,189266,NaN,CASH SENT TO PERUMANALLU BRANCH TAKE OVER PURP...,0.0,800000.0,NaN,0182,TIRUPUR LAKSHMI NAGAR,40168221385,73792.6,DR,None,cr


In [9]:
REQUIRED_COLUMNS = [
    "txn_date", "vh_no", "type", "chq_no", "cust_name", "particulars", "credit_amount", "debit_amount", "balance_amount", "balance_type", "branch_code", "branch_name", "acc_no", "txn_type"
]

In [10]:
available_cols = [c for c in REQUIRED_COLUMNS if c in df.columns]
df = df[available_cols]
missing = set(REQUIRED_COLUMNS) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")
len(df.columns)

df.columns

Index(['txn_date', 'vh_no', 'type', 'chq_no', 'cust_name', 'particulars',
       'credit_amount', 'debit_amount', 'balance_amount', 'balance_type',
       'branch_code', 'branch_name', 'acc_no', 'txn_type'],
      dtype='object')

In [11]:
# Extract last balance for each account number
last_balance = (
    df.groupby("acc_no")["balance_amount"]
    .last()
    .reset_index()
    .rename(columns={"balance_amount": "closing_balance"})
)

# Merge into dim_balance
dim_rec_df = dim_rec_df.merge(last_balance, on="acc_no", how="left")

dim_rec_df["label"] = "Ledger"
dim_rec_df

,branch_code,branch_name,acc_no,opening_balance,closing_balance,label
0,0182,TIRUPUR LAKSHMI NAGAR,40168221385,353857.60,2162209.00,Ledger
1,0627,HYDERABAD VANASTALIPURAM,43143793734,2759866.14,7749226.14,Ledger
2,1046,UTHUKOTTAI,43782239368,2092603.80,344678.80,Ledger
3,4304,GAJWEL (AP),33069117110,400014.04,1125808.06,Ledger


In [64]:
dim_rec_df.columns

Index(['branch_code', 'branch_name', 'acc_no', 'opening_balance',
       'closing_balance'],
      dtype='object')

In [ ]:
# df.to_csv("cleaned_ledger_fact.csv", index=False)

In [1]:
#--- Database connection test ---
from sqlalchemy import create_engine, inspect
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host":     os.getenv("DB_HOST"),
        "port":     os.getenv("DB_PORT"),
        "dbname":   os.getenv("DB_NAME"),
        "user":     os.getenv("DB_USER"),
        "password": os.getenv("DB_PASSWORD"),
    }
)

# Test the connection
with engine.connect() as conn:
    print("✅ Connected to PostgreSQL successfully!")

✅ Connected to PostgreSQL successfully!


In [107]:
# List tables in the public schema
inspector = inspect(engine)
tables = inspector.get_table_names(schema="public")

print(tables)

['ledger_fact', 'ledger_balance', 'bank_fact', 'bank_balance']


In [3]:
import pandas as pd

In [6]:
final_ledger_df = pd.read_csv("C:/Users/user/Documents/NESA/Muthoot/Data Cleaning/final_ledger/cleaned_ledger_fact1.csv",  encoding="latin1")
final_ledger_df.head()

,txn_date,vh_no,type,chq_no,cust_name,particulars,credit_amount,debit_amount,balance_amount,balance_type,branch_code,branch_name,acc_no,txn_type
0,02-03-2026,1,BV,189263.0,MAHALAKSHMI MAHALAKSHMI,Cheque :MAHALAKSHMI MAHALAKSHMI#BTB#BTB8-TN17#...,382075.0,0,28217.4,CR,182,TIRUPUR LAKSHMI NAGAR,40168221385,cr
1,02-03-2026,2,BV,NaN,"MR.NAVEEN,MR.GOPAL",EXCESS CASH SENT TO OUR BANK ACCOUNT THROUGH M...,0.0,1000000,971782.6,DR,182,TIRUPUR LAKSHMI NAGAR,40168221385,dr
2,03-03-2026,4,BV,189265.0,MR THANAGAPANDIAN,TAKEOVER CASH SENT TO VALAYANKADU BRANCH CUSTO...,35040.0,0,936742.6,DR,182,TIRUPUR LAKSHMI NAGAR,40168221385,cr
3,03-03-2026,5,BV,189264.0,MR ANAND,TAKEOVER CASH SENT TO VALAYANKADU BRANCH CUSTO...,62950.0,0,873792.6,DR,182,TIRUPUR LAKSHMI NAGAR,40168221385,cr
4,03-03-2026,6,BV,189266.0,NaN,CASH SENT TO PERUMANALLU BRANCH TAKE OVER PURP...,800000.0,0,73792.6,DR,182,TIRUPUR LAKSHMI NAGAR,40168221385,cr


In [7]:
#--- Insert data into PostgreSQL ---
final_ledger_df.to_sql(
    name = "ledger_fact",
    con = engine,
    if_exists = "replace",
    index = False,
    chunksize = 1000
)

print(f"✅ {len(final_ledger_df)} rows inserted successfully!")

✅ 202 rows inserted successfully!


In [114]:
import pandas as pd

# Read back from DB to confirm
result = pd.read_sql("SELECT * FROM ledger_fact LIMIT 5;", con=engine)
print((result))

     txn_date  vh_no type    chq_no                cust_name  \
0  02-03-2026      1   BV  189263.0  MAHALAKSHMI MAHALAKSHMI   
1  02-03-2026      2   BV       NaN                     None   
2  03-03-2026      4   BV  189265.0                     None   
3  03-03-2026      5   BV  189264.0                     None   
4  03-03-2026      6   BV  189266.0                     None   

                                         particulars  credit_amount  \
0  Cheque :MAHALAKSHMI MAHALAKSHMI#BTB#BTB8-TN17#...       382075.0   
1  EXCESS CASH SENT TO OUR BANK ACCOUNT THROUGH M...            0.0   
2  TAKEOVER CASH SENT TO VALAYANKADU BRANCH CUSTO...        35040.0   
3  TAKEOVER CASH SENT TO VALAYANKADU BRANCH CUSTO...        62950.0   
4  CASH SENT TO PERUMANALLU BRANCH TAKE OVER PURP...       800000.0   

   debit_amount  balance_amount balance_type  branch_code  \
0             0         28217.4           CR          182   
1       1000000        971782.6           DR          182   
2    

In [102]:
type(dim_rec_df)

pandas.core.frame.DataFrame

In [128]:
# Insert data into PostgreSQL
dim_rec_df.to_sql(
    name = "ledger_balance",
    con = engine,
    if_exists = "replace",
    index = False,
    method="multi"
)

print(f"✅ {len(dim_rec_df)} rows inserted successfully!")

✅ 4 rows inserted successfully!


In [129]:
result = pd.read_sql("SELECT * FROM ledger_balance LIMIT 5;", con=engine)
print(result)

  branch_code               branch_name       acc_no  opening_balance  \
0        0182     TIRUPUR LAKSHMI NAGAR  40168221385        353857.60   
1        0627  HYDERABAD VANASTALIPURAM  43143793734       2759866.14   
2        1046                UTHUKOTTAI  43782239368       2092603.80   
3        4304               GAJWEL (AP)  33069117110        400014.04   

   closing_balance   label  
0       2162209.00  Ledger  
1       7749226.14  Ledger  
2        344678.80  Ledger  
3       1125808.06  Ledger  
